# Foveance — try it in 30 seconds

**Cut your LLM token bill 60%+ without changing your code or your answers.**

Your AI agent's chat history piles up — you pay for every old message on every turn, and the model gets *worse* as facts get buried. Foveance keeps what matters and trims the rest.

This notebook runs entirely offline (no API key, no GPU). Just press **Runtime → Run all**.

In [ ]:
!pip install -q foveance

## The one-liner: `shrink(messages)`

Take a normal OpenAI-style `messages` list, and get back a shorter one — same system prompt, same latest turn, older turns intelligently compressed.

In [ ]:
from foveance import shrink

# a long agent history with one load-bearing fact buried early amid clutter
noise = "\n".join(f"log {i}: status=ok latency={i%400}ms path=/srv/{i%97}" for i in range(60))
messages = [
    {"role": "system", "content": "You are a deployment assistant. Be concise."},
    {"role": "user", "content": f"Deployment notes.\nAPI_KEY=SECRET-7431\n{noise}"},
    {"role": "assistant", "content": "Noted the deployment logs."},
    {"role": "user", "content": f"More logs (nothing important):\n{noise}"},
    {"role": "assistant", "content": "Still healthy."},
    {"role": "user", "content": "What was the API_KEY from the first message?"},
]

smaller = shrink(messages, budget=400)

def approx_tokens(msgs):
    import json
    return len(json.dumps(msgs)) // 4   # rough chars/4 estimate

print(f"before: ~{approx_tokens(messages):>5} tokens across {len(messages)} messages")
print(f"after : ~{approx_tokens(smaller):>5} tokens across {len(smaller)} messages")
print(f"saved : {100*(1-approx_tokens(smaller)/approx_tokens(messages)):.0f}%")
print()
print("last turn kept verbatim:", smaller[-1]["content"])
print("buried fact still present:", "SECRET-7431" in __import__('json').dumps(smaller))

Now you'd send `smaller` to your model instead of `messages` — same answer, a fraction of the tokens.

## The offline benchmark demo

See the token savings per policy across budgets (uses a deterministic mock model, so it runs anywhere).

In [ ]:
!foveance demo --turns 24 --budgets 800,1600,2500

## Use it with a real agent (no code changes)

On your own machine, route any tool through the proxy with one command:

```bash
pip install foveance
foveance wrap claude          # or: foveance wrap -- codex "fix the tests"
```

It prints how many tokens you saved when you exit, and a live dashboard runs at http://localhost:8799/.

---

- **Repo:** https://github.com/aimaghsoodi/foveance
- **Docs:** https://aimaghsoodi.github.io/foveance/
- **PyPI:** `pip install foveance`

If this is useful, a ⭐ on the repo helps a lot!